# handlers

> Workflow-specific alignment handlers: init

In [ ]:
#| default_exp routes.handlers

In [ ]:
#| export
from typing import Any, Dict, List, Tuple, Callable, NamedTuple

from fasthtml.common import Span, APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

# Local imports (page-specific, no cross-package dependencies)
from cjm_transcript_vad_align.html_ids import AlignmentHtmlIds
from cjm_transcript_vad_align.models import VADChunk, AlignmentUrls
from cjm_transcript_vad_align.services.alignment import AlignmentService

# Alignment components (same package)
from cjm_transcript_vad_align.components.step_renderer import (
    render_align_column_body, render_align_mini_stats_text,
)

# Route core helpers (same package)
from cjm_transcript_vad_align.routes.core import (
    WorkflowStateStore, _get_selection_state, _update_alignment_state,
)

# Source service for fetching media_path
from cjm_transcript_source_select.services.source import SourceService

# Debug flag for alignment data flow tracing (set False in production)
DEBUG_ALIGNMENT = True

## Initialize Handler

Fetches VAD data from the alignment service and stores in state.

In [ ]:
#| export
class AlignInitResult(NamedTuple):
    """Result from pure alignment init handler.
    
    Contains domain-specific data for the combined layer wrapper to use
    when building cross-domain OOB elements (shared chrome, alignment status).
    """
    column_body: Any  # Rendered column body content
    chunks: List[VADChunk]  # Initialized VAD chunks
    focused_index: int  # Focused chunk index (always 0 on init)
    visible_count: int  # Visible card count
    card_width: int  # Card stack width in rem
    media_path: str  # Path to audio file (may be None)

In [ ]:
#| export
async def _handle_align_init(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    source_service:SourceService,  # Service for fetching source blocks
    alignment_service:AlignmentService,  # Service for VAD analysis
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:AlignmentUrls,  # URL bundle
    visible_count:int=DEFAULT_VISIBLE_COUNT,  # Initial visible card count
    card_width:int=DEFAULT_CARD_WIDTH,  # Initial card width in rem
) -> AlignInitResult:  # Pure domain result for wrapper to use
    """Initialize alignment from audio file via VAD plugin.
    
    Returns pure domain data. The combined layer wrapper adds cross-domain
    coordination (shared chrome, alignment status).
    """
    session_id = get_session_id(sess)

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Starting alignment initialization")
        print(f"[ALIGN_INIT] session_id: {session_id}")

    # Get media_path from selected sources (Phase 1)
    selection_state = _get_selection_state(state_store, workflow_id, session_id)
    selected_sources = selection_state.get("selected_sources", [])

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] selected_sources count: {len(selected_sources)}")

    # Extract media_path from first selected source's source block
    media_path = None
    if selected_sources:
        first_source = selected_sources[0]
        block = source_service.get_transcription_by_id(
            record_id=first_source["record_id"],
            provider_id=first_source["provider_id"],
        )
        if DEBUG_ALIGNMENT and block:
            print(f"[ALIGN_INIT] block.media_path: {block.media_path}")
        if block:
            media_path = block.media_path

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] extracted media_path: {media_path}")

    # Fetch VAD data
    chunks = []
    audio_duration = 0.0
    if media_path and alignment_service.is_available():
        if DEBUG_ALIGNMENT:
            print(f"[ALIGN_INIT] Calling VAD plugin with media_path: {media_path}")
        chunks, audio_duration = await alignment_service.analyze_audio_async(media_path)
        if DEBUG_ALIGNMENT:
            print(f"[ALIGN_INIT] VAD returned {len(chunks)} chunks, duration: {audio_duration:.2f}s")

    # Serialize and store
    chunk_dicts = [c.to_dict() for c in chunks]
    _update_alignment_state(
        state_store, workflow_id, session_id,
        vad_chunks=chunk_dicts,
        focused_chunk_index=0,
        is_initialized=True,
        visible_count=visible_count,
        card_width=card_width,
        media_path=media_path,
        audio_duration=audio_duration,
    )

    # Render column body (Web Audio API handles accurate seeking)
    column_body = render_align_column_body(
        chunks=chunks,
        focused_index=0,
        visible_count=visible_count,
        card_width=card_width,
        urls=urls,
        kb_system=None,
        media_path=media_path,
    )

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Returning AlignInitResult")

    return AlignInitResult(
        column_body=column_body,
        chunks=chunks,
        focused_index=0,
        visible_count=visible_count,
        card_width=card_width,
        media_path=media_path,
    )

## Router Initialization

Creates the workflow router with the init route.

In [ ]:
#| export
def init_workflow_router(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    source_service:SourceService,  # Service for fetching source blocks
    alignment_service:AlignmentService,  # Service for VAD analysis
    prefix:str,  # Route prefix (e.g., "/workflow/align/workflow")
    urls:AlignmentUrls,  # URL bundle (populated after routes defined)
    handler_init:Callable=None,  # Optional wrapped init handler
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize workflow routes for alignment.
    
    Accepts optional handler override for wrapping with cross-domain
    coordination (e.g., shared chrome, alignment status OOB updates).
    """
    router = APIRouter(prefix=prefix)

    # Use provided handler or fall back to raw domain handler
    _init = handler_init or _handle_align_init

    # -------------------------------------------------------------------------
    # Workflow Operations
    # -------------------------------------------------------------------------

    @router
    async def init(request, sess):
        """Initialize alignment from audio file via VAD plugin."""
        return await _init(
            state_store, workflow_id, source_service, alignment_service,
            request, sess, urls=urls,
        )

    # -------------------------------------------------------------------------
    # Route Dict
    # -------------------------------------------------------------------------

    routes = {
        "init": init,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()